The plan for this file it to have all the code to go from the different planet spectra to them making a new planet list hopefully in an organsied way so it can all be done from one file. 


In [ ]:
#--- Imports ---#
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from pathlib import Path
import emcee
#--- Data sheets ---# 

Planet_data= pd.read_csv('Data/planets.csv', comment='#')



# --- Classes ---#

class Config:
    def __init__(self):
        self.data_path= Path(r"Group 1 Full Loop Code\165 planets data\pandexo csv files\results20260212_163633.csv")
        self.data= pd.read_csv(self.data_path, comment='#')
        self.T_eq= self.data['Equilibrium_Temperature']
        self.T_eq_err= self.data['Uncertainty_on_Equilibrium_Temperature']
        self.Feture_height= self.data[]

class Model:
    def __init__(self, config: Config = None):
        self.data= config.data if config else None
        
    
class Statistics:
    def __init__(self, config: Config = None):
        self.data= config.data if config else None
        self.model = lambda T, A, B, C: A*T**2 + B*T + C
        self.Quad_priors= ((-30, 30), (-30, 30), (0, 40))
        self.Linear_priors= ((-20, 20), (-20, 50))
        self.nwalkers= 64
        self.Quad_ititails=( (np.random.uniform(-1e-2, 1e-2, self.nwalkers) ),(np.random.uniform(-3, 3, self.nwalkers)))
        self.Lin_ititials=((-1e-2, 1e-2), (-3, 3))
        
    def Quadratic_stuff(self):

        def quadratic_model(self, T, A, B, C):
            return A*T**2 + B*T + C
        
        def log_likelihood(self, theta, T, y, yerr):
            A, B, C = theta
            model = quadratic_model(self,T, A, B, C)
            return -0.5 * np.sum(
                (y - model)**2 / yerr**2 + np.log(2*np.pi*yerr**2)
            )
        
        def log_prior(self, theta):
            A, B, C = theta
            if self.Quad_priors[0][0] < A < self.Quad_priors[0][1] and self.Quad_priors[1][0] < B < self.Quad_priors[1][1] and self.Quad_priors[2][0] < C < self.Quad_priors[2][1]:
                return 0.0
            return -np.inf

        def log_probability(self, theta, T, y, yerr):
            lp = log_prior(self,theta)
            if not np.isfinite(lp):
                return -np.inf
            return lp + log_likelihood(self,theta, T, y, yerr)
        


    def Linear_stuff(self):

        def linear_model(self, T, B, C):
            return B*T + C


        def log_likelihood_linear(self, theta, T, y, yerr):
            B, C = theta
            model = linear_model(self,T, B, C)
            return -0.5 * np.sum(
                (y - model)**2 / yerr**2 + np.log(2*np.pi*yerr**2)
            )


        def log_prior_linear(self, theta):
            B, C = theta
            if self.Linear_priors[0][0] < B < self.Linear_priors[0][1] and self.Linear_priors[1][0] < C < self.Linear_priors[1][1]:
                return 0.0
            return -np.inf


        def log_probability_linear(self, theta, T, y, yerr):
            lp = log_prior_linear(self,theta)
            if not np.isfinite(lp):
                return -np.inf
            return lp + log_likelihood_linear(self,theta, T, y, yerr)
    
    def Run_MCMC(self, T, y, yerr,ndim,  nsteps=9000):
        pos = np.zeros((self.nwalkers, ndim))
        for i in range(ndim):
            pos[:, i] = np.random.uniform(-1e-2, 1e-2, self.nwalkers)   # A
            pos[:, 1] = np.random.uniform(-3, 3, self.nwalkers)        # B
            pos[:, 2] = np.random.normal(np.mean(y), 0.2*np.std(y), self.nwalkers)

    def run_mcmc_linear(self,T, y, yerr, nsteps=9000):
        ndim, nwalkers = 2, 64

        pos = np.zeros((nwalkers, ndim))
        pos[:, 0] = -2.0 + 1e-3*np.random.randn(nwalkers)   # B
        pos[:, 1] = np.random.uniform(5, 45, nwalkers)  # C

        sampler = emcee.EnsembleSampler(
            nwalkers, ndim, self.Linear_stuff.log_probability_linear, args=(T, y, yerr)
        )
        sampler.run_mcmc(pos, nsteps, progress=False)
        
        burn=1000
        tau_lin = sampler.get_autocorr_time(discard=burn, thin=1)
        samples = sampler.get_chain(discard=1000, thin=1, flat=True)
        return samples, tau_lin

    def run_mcmc(T, y, yerr, nsteps=9000):
        ndim, nwalkers = 3, 64

        pos = np.zeros((nwalkers, ndim))
        pos[:, 0] = np.random.uniform(-1e-2, 1e-2, nwalkers)   # A
        pos[:, 1] = np.random.uniform(-3, 3, nwalkers)        # B
        pos[:, 2] = np.random.normal(np.mean(y), 0.2*np.std(y), nwalkers)

        sampler = emcee.EnsembleSampler(
            nwalkers, ndim, self.Quadratic_stuff.log_probability, args=(T, y, yerr)
        )
        sampler.run_mcmc(pos, nsteps, progress=True)

        burn = 1000
        tau_quad = sampler.get_autocorr_time(discard=burn, thin=1)
        samples = sampler.get_chain(discard=1000, thin=1, flat=True)
        return samples, tau_quad

    def run_mcmc(ln_prob, ndim, p0, nwalkers=32, burn=1000, steps=2000):
        sampler = emcee.EnsembleSampler(nwalkers, ndim, ln_prob, args=(x_s, A_H, A_H_err))
        state = sampler.run_mcmc(p0, burn, progress=True)
        sampler.reset()
        sampler.run_mcmc(state, steps, progress=True)
        return sampler



In [ ]:
# -----------------------------
# 1. Model + likelihood
# -----------------------------

def quadratic_model(T, A, B, C):
    return A*T**2 + B*T + C
    
def log_likelihood(theta, T, y, yerr):
    A, B, C = theta
    model = quadratic_model(T, A, B, C)
    return -0.5 * np.sum(
        (y - model)**2 / yerr**2 + np.log(2*np.pi*yerr**2)
    )
    
def log_prior(theta):
    A, B, C = theta
    if -30 < A < 30 and -30 < B < 30 and 0 < C < 40:
        return 0.0
    return -np.inf

def log_probability(theta, T, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, T, y, yerr)

# -----------------------------
# 2. MCMC runner
# -----------------------------

def run_mcmc(T, y, yerr, nsteps=9000):
    ndim, nwalkers = 3, 64

    pos = np.zeros((nwalkers, ndim))
    pos[:, 0] = np.random.uniform(-1e-2, 1e-2, nwalkers)   # A
    pos[:, 1] = np.random.uniform(-3, 3, nwalkers)        # B
    pos[:, 2] = np.random.normal(np.mean(y), 0.2*np.std(y), nwalkers)

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_probability, args=(T, y, yerr)
    )
    sampler.run_mcmc(pos, nsteps, progress=True)

    burn = 1000
    tau_quad = sampler.get_autocorr_time(discard=burn, thin=1)
    samples = sampler.get_chain(discard=1000, thin=1, flat=True)
    return samples, tau_quad

# -----------------------------
# 3. Linear model + BIC comparison
# -----------------------------

def linear_model(T, B, C):
    return B*T + C


def log_likelihood_linear(theta, T, y, yerr):
    B, C = theta
    model = linear_model(T, B, C)
    return -0.5 * np.sum(
        (y - model)**2 / yerr**2 + np.log(2*np.pi*yerr**2)
    )


def log_prior_linear(theta):
    B, C = theta
    if -20 < B < 20 and -20 < C < 50:
        return 0.0
    return -np.inf


def log_probability_linear(theta, T, y, yerr):
    lp = log_prior_linear(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_linear(theta, T, y, yerr)


def run_mcmc_linear(T, y, yerr, nsteps=9000):
    ndim, nwalkers = 2, 64

    pos = np.zeros((nwalkers, ndim))
    pos[:, 0] = -2.0 + 1e-3*np.random.randn(nwalkers)   # B
    pos[:, 1] = np.random.uniform(5, 45, nwalkers)  # C

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_probability_linear, args=(T, y, yerr)
    )
    sampler.run_mcmc(pos, nsteps, progress=False)
    
    burn=1000
    tau_lin = sampler.get_autocorr_time(discard=burn, thin=1)
    samples = sampler.get_chain(discard=1000, thin=1, flat=True)
    return samples, tau_lin


#center T values for better convergence
T0 = np.mean(Teq)
Tscale = np.std(Teq)
Tc = (Teq - T0) / Tscale
print(Tc.min(), Tc.max())


#Run MCMC for Quadratic

samples_quad, tau_quad = run_mcmc(
    T=Tc,
    y=feature_height,
    yerr=feature_height_err
)
#Run MCMC for Linear
samples_lin, tau_lin = run_mcmc_linear(
    T=Tc,
    y=feature_height,
    yerr=feature_height_err
)

print("Quadratric Autocorrelation times:", tau_quad)
print("Quadratic Max τ:", np.max(tau_quad))

print("Linear Autocorrelation times:", tau_lin)
print("Linear Max τ:", np.max(tau_lin))

import corner as corner
import matplotlib.pyplot as plt

# -----------------------------
# Quadratic corner plot
# -----------------------------

labels_quad = [
    r"$A$ (curvature)",
    r"$B$ (linear term)",
    r"$C$ (offset)"
]

fig_quad = corner.corner(
    samples_quad,
    labels=labels_quad,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt=".3g",
    title_kwargs={"fontsize": 12}
)

fig_quad.suptitle("Quadratic model posterior", fontsize=14)
plt.show()

# -----------------------------
# Linear corner plot
# -----------------------------

labels_lin = [
    r"$B$ (slope)",
    r"$C$ (offset)"
]

fig_lin = corner.corner(
    samples_lin,
    labels=labels_lin,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt=".3g",
    title_kwargs={"fontsize": 12}
)

fig_lin.suptitle("Linear model posterior", fontsize=14)
plt.show()

#Revert coefficients to uncentered T for easier interpretation

def revert_quadratic_from_centered(samples, T0, Tscale):
    """
    Convert quadratic coefficients fitted in centered+scaled temperature
    back to raw temperature coefficients.

    Tc = (T - T0) / Tscale
    y  = Ac * Tc^2 + Bc * Tc + Cc

    Returns coefficients A, B, C such that:
    y = A * T^2 + B * T + C
    """

    Ac = samples[:, 0]
    Bc = samples[:, 1]
    Cc = samples[:, 2]

    A = Ac / Tscale**2
    B = Bc / Tscale - 2.0 * Ac * T0 / Tscale**2
    C = Cc - Bc * T0 / Tscale + Ac * T0**2 / Tscale**2

    return np.column_stack([A, B, C])
    
samples_quad_uncentered = revert_quadratic_from_centered(
    samples_quad, T0, Tscale
)

def revert_linear_from_centered(samples, T0, Tscale):
    Bc = samples[:, 0]
    Cc = samples[:, 1]

    B = Bc / Tscale
    C = Cc - Bc * T0 / Tscale

    return np.column_stack([B, C])
    
samples_lin_uncentered = revert_linear_from_centered(
    samples_lin,
    T0,
    Tscale
)

# -----------------------------
# 8. Compute BICs
# -----------------------------

# Quadratic fit
A_med, B_med, C_med = np.median(samples_quad_uncentered, axis=0)
logL_quad = log_likelihood(
    [A_med, B_med, C_med],
    Teq,
    feature_height,
    feature_height_err
)

k_quad = 3
n = len(Teq)
BIC_quad = k_quad * np.log(n) - 2 * logL_quad

# Linear fit
B_lin, C_lin = np.median(samples_lin_uncentered, axis=0)

logL_lin = log_likelihood_linear(
    [B_lin, C_lin],
    Teq,
    feature_height,
    feature_height_err
)

k_lin = 2
BIC_lin = k_lin * np.log(n) - 2 * logL_lin

# -----------------------------
# 9. Report comparison
# -----------------------------

delta_BIC = BIC_quad - BIC_lin

print(f"BIC (quadratic) = {BIC_quad:.2f}")
print(f"BIC (linear)    = {BIC_lin:.2f}")
print(f"ΔBIC (quad − lin) = {delta_BIC:.2f}")

if delta_BIC < -10:
    verdict = "Very strong evidence for curvature (quadratic)"
elif delta_BIC < -6:
    verdict = "Strong evidence for curvature"
elif delta_BIC < -2:
    verdict = "Positive evidence for curvature"
elif delta_BIC < 2:
    verdict = "No meaningful preference"
else:
    verdict = "Linear model preferred"

print("Interpretation:", verdict)

#Plotting the linear fit with posterior samples

B_med, C_med = np.median(samples_lin_uncentered, axis=0)
B_err, C_err = np.std(samples_lin_uncentered, axis=0)

print(f"B = {B_med:.4f} ± {B_err:.4f}")
print(f"C = {C_med:.2f} ± {C_err:.2f}")

T_plot = np.linspace(Teq.min(), Teq.max(), 300)

plt.errorbar(
    Teq, feature_height, yerr=feature_height_err,
    fmt='o', color='black'
)

# Posterior cloud
idx = np.random.choice(len(samples_lin_uncentered), 100, replace=False)
for i in idx:
    B, C = samples_lin_uncentered[i]
    plt.plot(
        T_plot,
        B*T_plot + C,
        color='orange',
        alpha=0.15
    )

# Posterior median
plt.plot(
    T_plot,
    B_med*T_plot + C_med,
    color='blue', lw=2.5, label='Posterior median'
)
#plt.ylim(-300,-400)
plt.xlabel("Temperature")
plt.ylabel("Feature height")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#Plotting Quadratic fit with posterior samples

# Median and 1σ from posterior
A_med, B_med, C_med = np.median(samples_quad_uncentered, axis=0)
A_err, B_err, C_err = np.std(samples_quad_uncentered, axis=0)

print(f"A = {A_med:.6f} ± {A_err:.6f}")
print(f"B = {B_med:.4f} ± {B_err:.4f}")
print(f"C = {C_med:.2f} ± {C_err:.2f}")

T_plot = np.linspace(Teq.min(), Teq.max(), 300)

plt.figure()

# Data with error bars
plt.errorbar(
    Teq,
    feature_height,
    yerr=feature_height_err,
    fmt='o',
    color='black'
)

# Posterior cloud
idx = np.random.choice(len(samples_quad_uncentered), 100, replace=False)
for i in idx:
    A, B, C = samples_quad_uncentered[i]
    plt.plot(
        T_plot,
        A*T_plot**2 + B*T_plot + C,
        color='orange',
        alpha=0.15
    )

# Posterior median curve
plt.plot(
    T_plot,
    A_med*T_plot**2 + B_med*T_plot + C_med,
    color='blue',
    lw=2.5,
    label='Posterior median'
)

plt.xlabel("Temperature")
plt.ylabel("Feature height")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# ============================================================
# Constraining power (CENTERED temperature compatible)
# ============================================================

def rank_constraining_points_linear(Tc, yerr):
    """
    Rank points by how much they constrain the linear slope.
    Tc must be CENTERED temperature (Tc = T - <T>).
    """
    weights = 1.0 / yerr**2

    # Pivot is zero by construction for centered T
    T_pivot = 0.0

    power = weights * Tc**2
    order = np.argsort(power)[::-1]

    return order, power, T_pivot


def rank_constraining_points_quadratic(Tc, yerr):
    """
    Rank points by how much they constrain the quadratic curvature.
    Tc must be CENTERED temperature (Tc = T - <T>).
    """
    weights = 1.0 / yerr**2

    # Curvature information scales as Tc^4
    power = weights * Tc**4
    order = np.argsort(power)[::-1]

    return order, power


# ============================================================
# Usage
# ============================================================

# Center temperature (should already exist, but safe)
T0 = np.mean(Teq)
Tc = Teq - T0

# Linear constraining power
order_l, power_l, T_pivot = rank_constraining_points_linear(
    Tc, feature_height_err
)

# Quadratic constraining power
order_q, power_q = rank_constraining_points_quadratic(
    Tc, feature_height_err
)


# ============================================================
# Reporting
# ============================================================

print("===== TOP 5 CONSTRAINING POINTS (LINEAR SLOPE) =====")
print(f"Pivot temperature (Teq): {T0:.1f}")
for i in order_l[:5]:
    print(
        f"T = {Teq[i]:.1f}, "
        f"yerr = {feature_height_err[i]:.2f}, "
        f"power = {power_l[i]:.3e}"
    )

print("\n===== TOP 5 CONSTRAINING POINTS (QUADRATIC CURVATURE) =====")
for i in order_q[:5]:
    print(
        f"T = {Teq[i]:.1f}, "
        f"yerr = {feature_height_err[i]:.2f}, "
        f"power = {power_q[i]:.3e}"
    )






In [ ]:
#-----------------------------------------Abis cell-----------------------------

#---reading in the file from the github data

#causing problems with the column names, chat suggested this and it fixed the problem
#dont forget to put a space between ' and file name

#-------All model definitions and runnning MCMC


#------------ MCMC

import emcee

def run_mcmc(ln_prob, ndim, p0, nwalkers=32, burn=1000, steps=2000):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, ln_prob, args=(x_s, A_H, A_H_err))
    state = sampler.run_mcmc(p0, burn, progress=True)
    sampler.reset()
    sampler.run_mcmc(state, steps, progress=True)
    return sampler


nwalkers = 32

# linear
ndim_lin = 2
p0_lin_center = np.array([0.0, 0.0])
p0_lin = p0_lin_center + 1e-2*np.random.randn(nwalkers, ndim_lin)
sampler_lin = run_mcmc(ln_prob_linear, ndim_lin, p0_lin)

# quadratic
ndim_quad = 3
p0_quad_center = np.array([0.0, 0.0, 0.0])
p0_quad = p0_quad_center + 1e-2*np.random.randn(nwalkers, ndim_quad)
sampler_quad = run_mcmc(ln_prob_quad, ndim_quad, p0_quad)

import matplotlib.pyplot as plt

x_line = np.linspace(eq_temp.min(), eq_temp.max(), 300)
x_line_s = (x_line - x0)/xs

# sample a few posterior draws
s_lin = sampler_lin.get_chain(flat=True)
s_quad = sampler_quad.get_chain(flat=True)

plt.figure()
#plt.plot(x_line, y_med, color="navy", lw=3, label="Median fit")
plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt="o", alpha=0.4, label="data", color = 'slategray')

for theta in s_lin[np.random.randint(len(s_lin), size=50)]:
    plt.plot(x_line, linear_model(theta, x_line_s), alpha=0.15, color = 'blue')

#for theta in s_quad[np.random.randint(len(s_quad), size=50)]:
#    plt.plot(x_line, quad_model(theta, x_line_s), alpha=0.15, color = 'purple')

plt.xlabel("x")
plt.ylabel("A_H")
plt.ylim(-100,200)
plt.title("Linear model")
plt.legend()
plt.show()

#plt.plot(x_line, y_med_quad, color="darkred", lw=3, label="Median fit")
plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt="o", alpha=0.4, label="data", color = 'slategray')
for theta in s_quad[np.random.randint(len(s_quad), size=50)]:
    plt.plot(x_line, quad_model(theta, x_line_s), alpha=0.15, color = 'purple')

plt.xlabel("x")
plt.ylabel("A_H")
plt.title("Quadratic model")
plt.ylim(-100, 200)
plt.legend()
plt.show()

#-------------------BIC calculation

theta_lin = [0.5, 20]
theta_quad = [-0.05, -2, 13]

lnL_lin = ln_likelihood(linear_model, theta_lin, eq_temp, A_H, A_H_err)
lnL_quad = ln_likelihood(quad_model, theta_quad, eq_temp, A_H, A_H_err)

k_lin = 2
k_quad = 3
n = len(A_H)

BIC_lin = k_lin * np.log(n) - 2 * (lnL_lin)
print(f"linear BIC value:", BIC_lin)
BIC_quad = k_quad * np.log(n) - 2 * (lnL_quad)
print(f"quadratic BIC value:",BIC_quad)   
delta_BIC = BIC_quad - BIC_lin
print("-------")
print(f"delta BIC:", delta_BIC)

best_BIC = "Linear" if BIC_lin < BIC_quad else "Quadratic"
print("-------")
print(best_BIC, f"is the model to be favoured")

#relative likelihood
rel_like = np.exp(-0.5*delta_BIC)
print(f"the relative likelihood is:", rel_like)
#this should be fine but BIC is very large

#----------------corner plots, autocorrection time and liklihood values

#linear 

import corner

labels_lin = ["m", "b"]

flat_samples_lin = sampler_lin.get_chain(discard=1000, thin=1, flat=True)

fig = corner.corner(
    flat_samples_lin,
    labels=labels_lin,
    show_titles=True
)

imax = np.argmax(flat_samples_lin)

m_ml, b_ml = flat_samples_lin[imax]
#s_int_ml = np.exp(log_s_ml)

print("Best sample (MAP):")
print("m =", m_ml)
print("b =", b_ml)
#print("log_s =", log_s_ml)
#print("sigma_int =", s_int_ml)


print("----")
#autocorrection time
tau = sampler_lin.get_autocorr_time()
print(f"autocorrection time is:")
print(tau)
#flat_samples = sampler_lin.get_chain(discard=1000, thin=1, flat=True)
print(flat_samples_lin.shape)

# quadratic
import numpy as np
import corner

labels_quad = ["a", "b", "c"]

# chain and log-prob, flattened the same way
flat_samples_quad = sampler_quad.get_chain(discard=1000, thin=1, flat=True)
flat_logprob_quad = sampler_quad.get_log_prob(discard=1000, thin=1, flat=True)

fig = corner.corner(
    flat_samples_quad[:, :3],   # in case you ever have extra params later
    labels=labels_quad,
    show_titles=True
)

# MAP / "best" sample = highest posterior (log-prob)
imax_Q = np.argmax(flat_logprob_quad)
a_ml, b_ml, c_ml = flat_samples_quad[imax_Q, :3]

print("Best sample (MAP):")
print("a =", a_ml)
print("b =", b_ml)
print("c =", c_ml)

print("----")

#--------------autocorrelation time (this can fail if chain is too short; handle gracefully)
try:
    tau = sampler_quad.get_autocorr_time()
    print("autocorrelation time is:")
    print(tau)
except Exception as e:
    print("autocorr time couldn't be estimated reliably (chain may be too short).")
    print("Error:", e)

print("flat_samples_quad shape:", flat_samples_quad.shape)
print("flat_logprob_quad shape:", flat_logprob_quad.shape)

#--------------posterior fits

plt.figure(figsize=(8,5))

# x grid for smooth lines (RAW axis)
x_grid = np.linspace(eq_temp.min(), eq_temp.max(), 300)
x_grid_s = (x_grid - x0) / xs   # scaled version for the model

inds = np.random.randint(len(flat_samples_lin), size=300)

for ind in inds:
    m, b = flat_samples_lin[ind][:2]
    y_line = m * x_grid_s + b
    plt.plot(x_grid, y_line, color="steelblue", alpha=0.10, lw=1)

plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt=".", capsize=0,
             color="slategray", alpha=0.35, label="data")

plt.xlabel("eq_temp")
plt.ylabel("A_H")
plt.title("Linear posterior fits")
plt.ylim(-100,200)
plt.legend()
plt.show()

indsQ = np.random.randint(len(flat_samples_quad), size=300)

for ind in indsQ:
    m, b = flat_samples_quad[ind][:2]
    y_line = a * (x_grid_s)**2 + b*(x_grid_s) + c
    plt.plot(x_grid, y_line, color="steelblue", alpha=0.10, lw=1)

plt.errorbar(eq_temp, A_H, yerr=A_H_err, fmt=".", capsize=0,
             color="slategray", alpha=0.35, label="data")

plt.xlabel("eq_temp")
plt.ylabel("A_H")
plt.title("Quadratic posterior fits")
plt.ylim(-100,200)
plt.legend()
plt.show()

#------------(reduced) chi squared

import numpy as np

# theta_lin: [m, b] or [m, b, log_s]
# theta_quad: [a, b, c] or [a, b, c, log_s]
# models take (theta, x_s) and return y_model

def chi_squared(model, theta, x_s, y, yerr):
    y_model = model(theta, x_s)
    return np.sum(((y - y_model) / yerr)**2)

def reduced_chi_squared(model, theta, x_s, y, yerr, n_params):
    chi2 = chi_squared(model, theta, x_s, y, yerr)
    dof = len(y) - n_params
    return chi2, chi2 / dof

# Use your actual data arrays
y = A_H
yerr = A_H_err

# number of fitted parameters (exclude log_s if you’re using it as a nuisance parameter)
k_lin  = 2
k_quad = 3

chi2_lin, red_chi2_lin   = reduced_chi_squared(linear_model, theta_lin,  x_s, y, yerr, k_lin)
chi2_quad, red_chi2_quad = reduced_chi_squared(quad_model,   theta_quad, x_s, y, yerr, k_quad)

print("linear chi2:", chi2_lin)
print("quadratic chi2:", chi2_quad)
print("reduced chi2 (linear, quad):", red_chi2_lin, red_chi2_quad)

# Bar plot for BIC and reduced chi-squared
fig, ax1 = plt.subplots(figsize=(6,4))

# BIC
ax1.bar(['Linear', 'Quadratic'], [BIC_lin, BIC_quad], color=['skyblue','salmon'], alpha=0.7)
ax1.set_ylabel("BIC", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Overlay reduced chi-squared on second axis
ax2 = ax1.twinx()
ax2.bar(['Linear', 'Quadratic'], [red_chi2_lin, red_chi2_quad], color=['lightgreen','orange'], alpha=0.4)
ax2.set_ylabel("Reduced χ²", color='green')
ax2.tick_params(axis='y', labelcolor='green')

plt.title("Comparison: BIC vs Reduced χ²")
#plt.ylim(0, 2e8)
plt.show()

In [ ]:
# --- Tamzin :) ---

# I have written my code in Google Colab, so the way files are imported will be different and may not work in VS code.
# This is easy to change, so if you have any problems running the code let me know and I will fix it!!
# I also tend to work with xlsx files rather than csv files (just personal preference), this is also easy to change!!

# First, I wrote code for a log-shaped MCMC-fitted curve based on the shape of the TSM vs Teq plot I made previously.
# I assumed that the shape of TSM vs Teq would be roughly the same as A_H vs Teq, I learned this was wrong later, but this is still useful code to have.
# For this code to run, I need to include the code I wrote to:
  # Use the planets spreadsheet.
  # Calculate TSM and Teq from the spreadsheet's data (with errors).
  # Put planets into the original temperature bins.
  # Choose the 5 planets with the highest TSMs in each temperature bin and fit a curve to them.

# ------------------------------
# START OF LOG-SHAPED MCMC CODE


# Fitting a curve using MCMC with intrinsic scatter and x errors AND chi squared test


def log_likelihood(theta, x, y, xerr, yerr):
    a, b, log_sigma_int = theta
    sigma_int = np.exp(log_sigma_int)

    model = a * x + b

    sigma_tot = np.sqrt(
        yerr**2
        + (a * xerr)**2
        + sigma_int**2)

    return -0.5 * np.sum(
        ((y - model) / sigma_tot)**2
        + np.log(2 * np.pi * sigma_tot**2))

def log_prior(theta):
    a, b, log_sigma_int = theta

    if (
        -5 < a < 5 and
        -20 < b < 20 and
        -10 < log_sigma_int < 1
    ):
        return 0.0

    return -np.inf

def log_posterior(theta, x, y, xerr, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y, xerr, yerr)

ndim, nwalkers = 3, 32
initial = np.array([0.0, 0.0, -2.0])  # start with small intrinsic scatter
pos = initial + 1e-3 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers,
    ndim,
    log_posterior,
    args=(x, y, xerr, yerr)
)
sampler.run_mcmc(pos, 5000, progress=True)

samples = sampler.get_chain(discard=1000, flat=True)

a_mcmc, b_mcmc, log_sigma_int_mcmc = np.median(samples, axis=0)
sigma_int_mcmc = np.exp(log_sigma_int_mcmc)

print(f"a = {a_mcmc:.3f}")
print(f"b = {b_mcmc:.3f}")
print(f"intrinsic scatter (dex) = {sigma_int_mcmc:.3f}")

# Generate fitted curve
Teq_range = np.linspace(sample_teq_vals.min(), sample_teq_vals.max(), 500)
x_range = np.log10(Teq_range)

y_model = a_mcmc * x_range + b_mcmc
mcmc_TSM = 10**y_model

# Chi squared test
y_model_data = a_mcmc * x + b_mcmc

sigma_tot = np.sqrt(yerr**2 + (a_mcmc * xerr)**2 + sigma_int_mcmc**2)

chi2 = np.sum(((y - y_model_data) / sigma_tot)**2)

ndof = len(y) - 3

chi2_red = chi2 / ndof

print(f"Chi-squared = {chi2:.3f}")
print(f"Degrees of freedom = {ndof}")
print(f"Reduced chi-squared = {chi2_red:.3f}")

# Plot (log)
plt.errorbar(T_eq_calc, TSM, yerr = TSM_err, xerr = T_eq_err, linestyle = 'none', fmt = ".", color = "grey", alpha = 0.5)
plt.errorbar(sample_teq_vals, sample_TSM_vals, xerr = sample_teq_vals_errs, yerr = sample_TSM_vals_errs, fmt=".", linestyle="none")
plt.plot(Teq_range, mcmc_TSM, color="black")
#plt.xscale("log")
plt.yscale("log")
plt.xlabel("Equilibrium Temperature (K)")
plt.ylabel("TSM")
plt.title("TSM vs Teq for High-TSM Planets (Log)")
plt.text(0.95, 0.05, f"$\\chi^2_\\mathrm{{red}}$ = {chi2_red:.2f}",
                     transform=plt.gca().transAxes,
                     verticalalignment='bottom',
                     horizontalalignment = 'right',
                     fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.tight_layout()
plt.show()

# Plot (linear)
plt.errorbar(T_eq_calc, TSM, yerr = TSM_err, xerr = T_eq_err, linestyle = 'none', fmt = ".", color = "grey", alpha = 0.5)
plt.errorbar(sample_teq_vals, sample_TSM_vals, xerr = sample_teq_vals_errs, yerr = sample_TSM_vals_errs, fmt=".", linestyle="none")
plt.plot(Teq_range, mcmc_TSM, color="black")
plt.xlabel("Equilibrium Temperature (K)")
plt.ylabel("TSM")
plt.title("TSM vs Teq for High-TSM Planets (Linear)")
plt.text(0.95, 0.95, f"$\\chi^2_\\mathrm{{red}}$ = {chi2_red:.2f}",
                     transform=plt.gca().transAxes,
                     verticalalignment='top',
                     horizontalalignment = 'right',
                     fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.ylim(0, 425)
plt.tight_layout()
plt.show()

print(f"Fitted relation:\nlog10(TSM) = {a_mcmc:.2f}log10(Teq) + {b_mcmc:.2f}\nTSM = {10**b_mcmc:.2f} × Teq^{a_mcmc:.2f}")



# END OF LOG-SHAPED MCMC CODE
# ----------------------------

# Next, I fit a quadratic curve to simulated data using MCMC.
# This does not rely on any data from spreadsheets.
# I included:
  # Chi-squared and reduced chi-squared tests.
  # Mean absolute error.
  # Posterior predictive checks (PPCs) on:
    # RMS (checks that scatter of data is reasonable given that the fitted model is true)
    # Curvature (checks that the shape of the data is reasonable given that the fitted model is true)

# ----------------------------
# START OF QUADRATIC MCMC CODE



# Replace x with Teq later
x = np.linspace(100, 2700, 30)

# True quadratic parameters
A_true = 1.5e-4
x0_true = 1500.0
ymin_true = 40.0

# Replace y with
y_true = A_true * (x - x0_true)**2 + ymin_true

# Uncertainties
xerr = 0.05 * x # 5% x uncertainty
yerr = 0.10 * y_true # 10% y uncertainty

# Add noise and intrinsic scatter
rng = np.random.default_rng(42)
sigma_int_true = 20.0

y_obs = (y_true + rng.normal(0, yerr) + rng.normal(0, sigma_int_true, size=len(x)))


# !!!!!!!!!!!!!!!!!!!!!!
# MCMC for the quadratic
# !!!!!!!!!!!!!!!!!!!!!!

def log_likelihood_quad(theta, x, y, yerr):
    A, x0, ymin, log_sigma_int = theta
    sigma_int = np.exp(log_sigma_int)
    model = A * (x - x0)**2 + ymin
    sigma_tot = np.sqrt(yerr**2 + sigma_int**2)

    return -0.5 * np.sum(
        ((y - model) / sigma_tot)**2
        + np.log(2 * np.pi * sigma_tot**2))


def log_prior_quad(theta):
    A, x0, ymin, log_sigma_int = theta
    if (
        -1e-3 < A < 1e-3 and
        0 < x0 < 3000 and
        0 < ymin < 500 and
        -10 < log_sigma_int < 5):
        return 0.0
    return -np.inf


def log_posterior_quad(theta, x, y, yerr):
    lp = log_prior_quad(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_quad(theta, x, y, yerr)

ndim_quad = 4
nwalkers = 32

initial_quad = np.array([A_true, x0_true, ymin_true, np.log(20.0)])
pos_quad = initial_quad + 1e-5 * np.random.randn(nwalkers, ndim_quad)

sampler_quad = emcee.EnsembleSampler(
    nwalkers, ndim_quad, log_posterior_quad, args=(x, y_obs, yerr)
)
sampler_quad.run_mcmc(pos_quad, 3000, progress=True)

samples_quad = sampler_quad.get_chain(discard=1000, flat=True)
log_probs_quad = sampler_quad.get_log_prob(discard=1000, flat=True)

lnL_max_quad = np.max(log_probs_quad)


# !!!!!!!!!!!!!!!!!
# MCMC for the line
# !!!!!!!!!!!!!!!!!

def log_likelihood_lin(theta, x, y, yerr):
    m, b, log_sigma_int = theta
    sigma_int = np.exp(log_sigma_int)
    model = m * x + b
    sigma_tot = np.sqrt(yerr**2 + sigma_int**2)

    return -0.5 * np.sum(
        ((y - model) / sigma_tot)**2
        + np.log(2 * np.pi * sigma_tot**2)
    )


def log_prior_lin(theta):
    m, b, log_sigma_int = theta
    if (
        -10 < m < 10 and
        -1000 < b < 1000 and
        -10 < log_sigma_int < 5
    ):
        return 0.0
    return -np.inf


def log_posterior_lin(theta, x, y, yerr):
    lp = log_prior_lin(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_lin(theta, x, y, yerr)

ndim_lin = 3

initial_lin = np.array([0.0, np.mean(y_obs), np.log(20.0)])
pos_lin = initial_lin + 1e-4 * np.random.randn(nwalkers, ndim_lin)

sampler_lin = emcee.EnsembleSampler(
    nwalkers, ndim_lin, log_posterior_lin, args=(x, y_obs, yerr)
)
sampler_lin.run_mcmc(pos_lin, 3000, progress=True)

samples_lin = sampler_lin.get_chain(discard=1000, flat=True)
log_probs_lin = sampler_lin.get_log_prob(discard=1000, flat=True)

lnL_max_lin = np.max(log_probs_lin)

x_fit = np.linspace(x.min(), x.max(), 500)
m_mcmc, b_mcmc, log_sigma_int_lin = np.median(samples_lin, axis=0)
y_fit_lin = m_mcmc * x_fit + b_mcmc


# !!!!!!!!!!!!!!!!!!!!!
# Finding BICs and ΔBIC
# !!!!!!!!!!!!!!!!!!!!!

n = len(y_obs)

# Will need to change this as needed later (will probably be 2 for line, 3 for quadratic)
k_quad = 4   # A, x0, ymin, sigma_int
k_lin = 3    # m, b, sigma_int

bic_quad = k_quad * np.log(n) - 2 * lnL_max_quad
bic_lin = k_lin * np.log(n) - 2 * lnL_max_lin

delta_bic = bic_lin - bic_quad

print(f"\nBIC (line) = {bic_lin:.2f}")
print(f"BIC (quadratic) = {bic_quad:.2f}")
print(f"ΔBIC = {delta_bic:.2f}")

A_mcmc, x0_mcmc, ymin_mcmc, log_sigma_int_mcmc = np.median(samples_quad, axis=0)

y_fit = A_mcmc * (x_fit - x0_mcmc)**2 + ymin_mcmc

#print(f"\nModel: y = {A_mcmc:.2f}(x - {x0_true:.2f})^2 + {ymin_mcmc:.2f}")


# !!!!!!!!!!!
# Chi-squared
# !!!!!!!!!!!

y_model_obs = A_mcmc * (x - x0_mcmc)**2 + ymin_mcmc
sigma_tot_obs = np.sqrt(yerr**2 + np.exp(log_sigma_int_mcmc)**2)

chi2 = np.sum(((y_obs - y_model_obs)/sigma_tot_obs)**2)
ndof = len(y_obs) - 4
chi2_red = chi2 / ndof

print(f"\nChi-squared = {chi2:.2f}")
print(f"Reduced chi-squared = {chi2_red:.3f}")


# !!!!!!!!!!!!!!!!!!!
# Mean Absolute Error
# !!!!!!!!!!!!!!!!!!!

residuals_quad = y_obs - y_model_obs

mae_quad = 1/len(residuals_quad) * sum(np.abs(residuals_quad))

print(f"\nMean Absolute Error (quadratic) = {mae_quad:.3f}")


# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# RMS Posterior Predictive Check (PPC)
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

idx = np.random.choice(len(samples_quad), size=500, replace=False)

rms_sim = []
for i in idx:
    A, x0, ymin, log_sigma = samples_quad[i]
    sigma_int = np.exp(log_sigma)

    y_sim = (
        A * (x - x0)**2 + ymin
        + rng.normal(0, yerr)
        + rng.normal(0, sigma_int, size=len(x)))

    y_model_i = A * (x - x0)**2 + ymin
    rms_sim.append(np.sqrt(np.mean((y_sim - y_model_i)**2)))

rms_obs = np.sqrt(np.mean(residuals_quad**2))
rms_obs_minus_sim = rms_obs - np.mean(rms_sim)
rms_z_score = rms_obs_minus_sim / np.std(rms_sim)

print(f"\nRMS PPC:")
print(f"Simulated RMS = {np.mean(rms_sim):.3f} ± {np.std(rms_sim):.3f}")
print(f"Observed RMS = {rms_obs:.3f}")
print(f"Observed - Simulated RMS = {rms_obs_minus_sim:.3f}")
print(f"Z-score = {rms_z_score:.3f}")


# !!!!!!!!!!!!!
# Curvature PPC
# !!!!!!!!!!!!!

def curvature_stat(y):
    """
    RMS of second finite differences.
    Measures curvature independent of slope/offset.
    """
    second_diff = y[2:] - 2*y[1:-1] + y[:-2]
    return np.sqrt(np.mean(second_diff**2))

T_curv_obs = curvature_stat(y_obs)

T_curv_sim = []

idx = np.random.choice(len(samples_quad), size=500, replace=False)

for i in idx:
    A, x0, ymin, log_sigma = samples_quad[i]
    sigma_int = np.exp(log_sigma)

    y_model_i = A * (x - x0)**2 + ymin

    y_sim = (
        y_model_i
        + rng.normal(0, yerr)
        + rng.normal(0, sigma_int, size=len(x))
    )

    T_curv_sim.append(curvature_stat(y_sim))

T_curv_sim = np.array(T_curv_sim)

curv_mean = np.mean(T_curv_sim)
curv_std = np.std(T_curv_sim)
curv_z = (T_curv_obs - curv_mean) / curv_std # less than or equal to 2 means shape is good

print(f"\nCurvature PPC:")
print(f"Simulated curvature = {curv_mean:.3f} ± {curv_std:.3f}")
print(f"Observed curvature = {T_curv_obs:.3f}")
print(f"Z-score = {curv_z:.3f}")


# Plotting
plt.errorbar(x, y_obs, yerr=yerr, fmt='.')
plt.plot(x_fit, y_fit, 'k-', label = "Quadratic")
plt.plot(x_fit, y_fit_lin, 'r-', label = "Line")
#plt.plot(x_fit, A_mcmc*(x_fit - x0_true)**2 + ymin_mcmc)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Simulated Quadratic Data to Test BIC Code")
plt.text(0.95, 0.95, f"Quadratic $\\chi^2_{{red}}$ = {chi2_red:.2f}\nQuadratic MAE = {mae_quad:.2f}",
         transform=plt.gca().transAxes,
         verticalalignment='top',
         horizontalalignment='right',
         fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.legend()
plt.tight_layout()
plt.show()



# END OF QUADRATIC MCMC CODE
# --------------------------